마우스로 쓴 숫자를 판단해봅시다.

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# 1. MNIST 데이터셋 로드 및 전처리
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 2. 모델 구성
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # (1, 28, 28) -> (32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                            # (32, 28, 28) -> (32, 14, 14)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),# (32, 14, 14) -> (64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                             # (64, 14, 14) -> (64, 7, 7)
            nn.Dropout2d(0.1)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x)


device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = SimpleCNN().to(device)

if not os.path.exists('mnist_model.pth'):
    # 3. 손실 함수 및 옵티마이저 설정
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())

    # 4. 모델 학습
    def train(model, device, train_loader, optimizer, epoch):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ' 
                    f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

    epochs = 5
    for epoch in range(1, epochs + 1):
        train(model, device, train_loader, optimizer, epoch)

    # 5. 모델 저장
    torch.save(model.state_dict(), 'mnist_model.pth')
    print("모델 저장 완료: mnist_model.pth")

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.305213
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.077823
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.109905
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.189088
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.059878
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.040079
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.034453
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.032998
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.069215
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.016934
Train Epoch: 2 [0/60000 (0%)]	Loss: 0.088418
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.017833
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.009902
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.079223
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.020909
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.068748
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.036943
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.110941
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.067442
Train Epoch: 2 [57600/60000 (96%)]	Loss: 0.027979
Train Epoch:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from ipycanvas import Canvas, hold_canvas
import numpy as np
from PIL import Image
import io

device = torch.device("cpu")
model = SimpleCNN().to(device) 
state_dict = torch.load("mnist_model.pth", map_location=device, weights_only=True)
model.load_state_dict(state_dict)

model.eval()

# 2. Canvas 설정
canvas = Canvas(width=200, height=200, sync_image_data=True)
canvas.fill_style = 'black'
canvas.fill_rect(0, 0, canvas.width, canvas.height)
canvas.stroke_style = 'white'
canvas.line_width = 12

drawing = False

def on_mouse_down(x, y):
    global drawing
    drawing = True
    canvas.begin_path()
    canvas.move_to(x, y)

def on_mouse_move(x, y):
    global drawing
    if drawing:
        canvas.line_to(x, y)
        canvas.stroke()

def on_mouse_up(x, y):
    global drawing
    drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

# 3. 예측 로직
from ipywidgets import Button, Output, VBox

btn_predict = Button(description="Predict")
btn_clear = Button(description="Clear")
out = Output()

def predict_digit(_):
    with out:
        out.clear_output()
        # 캔버스 데이터를 numpy 배열로 변환
        img_data = canvas.get_image_data()
        # RGBA에서 그레이스케일로 변환 및 전처리
        img = Image.fromarray(img_data).convert('L')
        img = img.resize((28, 28))
        
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        img_tensor = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(img_tensor)
            pred = output.argmax(dim=1, keepdim=True).item()
            print(f"결과: 이 숫자는 {pred}인 것 같네요!")

def clear_canvas(_):
    canvas.fill_style = 'black'
    canvas.fill_rect(0, 0, canvas.width, canvas.height)
    canvas.stroke_style = 'white'
    with out:
        out.clear_output()

btn_predict.on_click(predict_digit)
btn_clear.on_click(clear_canvas)

display(VBox([canvas, btn_predict, btn_clear, out]))